In [1]:
import pandas as pd
import numpy as np
import mindspore as ms
from mindspore import nn, Tensor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import moxing as mox

# Import model yang sudah dibuat di file lstm.py
from lstm import LSTMModel

ms.set_context(mode=ms.PYNATIVE_MODE)

INFO:root:Using MoXing-v2.1.0.5d9c87c8-5d9c87c8
INFO:root:Using OBS-Python-SDK-3.20.9.1


In [2]:
import moxing as mox
import pandas as pd

# 1. Tentukan path OBS dan path lokal tujuan di notebook
obs_path = "obs://mindspore-dataset-1/Data/data_bersih.csv"
local_path = "./data_bersih.csv"

# 2. Salin file dari OBS ke lokal notebook
print("Sedang mengunduh file dari OBS...")
mox.file.copy(obs_path, local_path)
print("Unduhan selesai!")

# 3. Baca data menggunakan pandas
df = pd.read_csv(local_path)

# 4. Tampilkan beberapa baris pertama data
df.head()

/home/ma-user/anaconda3/envs/MindSpore/lib/python3.7/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


Sedang mengunduh file dari OBS...
Unduhan selesai!


,Tanggal,Jam,Suhu,Kelembaban,CO2,NH3,Cahaya,Exhaust,Inline,Heater
0,12/06/2025,13:50,30.10,57.5,0.0,0.16566,22.2819,0.0,0.0,0.0
1,12/06/2025,17:00,29.50,74.9,0.0,0.52430,22.2819,0.0,0.0,0.0
2,12/06/2025,17:10,29.75,73.9,0.0,0.04455,22.2819,0.0,0.0,0.0
3,13/06/2025,13:30,32.00,55.1,0.0,0.08968,22.2819,0.0,0.0,1.0
4,13/06/2025,13:40,32.10,54.4,0.0,0.07164,22.2819,0.0,0.0,1.0


In [3]:
local_data_path = "./data_bersih.csv"
df = pd.read_csv(local_data_path).dropna()

df['Datetime'] = pd.to_datetime(
    df['Tanggal'] + ' ' + df['Jam'],
    format='%d/%m/%Y %H:%M'
)
df = df.sort_values('Datetime').reset_index(drop=True)

In [4]:
fitur_sensor = ['Suhu', 'Kelembaban', 'CO2', 'NH3', 'Cahaya']

# Smoothing rolling window
df[fitur_sensor] = df[fitur_sensor].rolling(window=3).mean()
df = df.dropna()

# Time features
df['hour_sin'] = np.sin(2 * np.pi * df['Datetime'].dt.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['Datetime'].dt.hour / 24)

fitur_input = fitur_sensor + ['hour_sin', 'hour_cos']
data = df[fitur_input].values

In [5]:
split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

scaler = MinMaxScaler()
scaler.fit(train_data)

train_scaled = scaler.transform(train_data)
test_scaled = scaler.transform(test_data)

In [6]:
split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

scaler = MinMaxScaler()
scaler.fit(train_data)

train_scaled = scaler.transform(train_data)
test_scaled = scaler.transform(test_data)

In [7]:
lookback = 18
forecast = 18  # 3 jam ke depan (asumsi data per 10 menit)

def create_dataset(data_scaled):
    X, y = [], []
    for i in range(len(data_scaled) - lookback - forecast + 1):
        X.append(data_scaled[i:i+lookback])
        y.append(data_scaled[i+lookback:i+lookback+forecast, :5]) # hanya ambil 5 kolom sensor utama
    return np.array(X), np.array(y)

X_train, y_train = create_dataset(train_scaled)
X_test, y_test = create_dataset(test_scaled)

# Flatten target untuk multi-output dense layer
y_train = y_train.reshape(len(y_train), -1)
y_test = y_test.reshape(len(y_test), -1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (3871, 18, 7)
y_train shape: (3871, 90)


In [8]:
# --- SETTING HYPERPARAMETER ---
HIDDEN_SIZE = 128      # Naikkan dari 64
DENSE_UNITS = 256      # Naikkan dari 128
LEARNING_RATE = 0.001 
EPOCHS = 120     
DROPOUT_PROB = 0.9    
BATCH_SIZE = 32
# ------------------------------

model = LSTMModel(
    input_size=len(fitur_input),
    hidden_size=HIDDEN_SIZE,
    dense_units=DENSE_UNITS,
    output_size=forecast * 5,
    keep_prob=DROPOUT_PROB
)

print(f"Model Baru berhasil diinisialisasi dengan 4 Hidden Layer!")

Model Baru berhasil diinisialisasi dengan 4 Hidden Layer!


In [9]:
loss_fn = nn.MSELoss()
optimizer = nn.Adam(model.trainable_params(), learning_rate=LEARNING_RATE)

net_with_loss = nn.WithLossCell(model, loss_fn)
train_step = nn.TrainOneStepCell(net_with_loss, optimizer)
train_step.set_train()

def train(X, y, epochs, batch_size):
    for epoch in range(epochs):
        total_loss = 0
        for i in range(0, len(X), batch_size):
            x_batch = Tensor(X[i:i+batch_size], ms.float32)
            y_batch = Tensor(y[i:i+batch_size], ms.float32)

            loss = train_step(x_batch, y_batch)
            total_loss += loss.asnumpy()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

print("Mulai training...")
train(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

Mulai training...
Epoch 1/120, Loss: 4.3827
Epoch 2/120, Loss: 1.5459
Epoch 3/120, Loss: 1.5145
Epoch 4/120, Loss: 1.5048
Epoch 5/120, Loss: 1.4910
Epoch 6/120, Loss: 1.4886
Epoch 7/120, Loss: 1.4954
Epoch 8/120, Loss: 1.5087
Epoch 9/120, Loss: 1.5405
Epoch 10/120, Loss: 1.5690
Epoch 11/120, Loss: 1.5637
Epoch 12/120, Loss: 1.5230
Epoch 13/120, Loss: 1.4982
Epoch 14/120, Loss: 1.4921
Epoch 15/120, Loss: 1.4824
Epoch 16/120, Loss: 1.4800
Epoch 17/120, Loss: 1.8393
Epoch 18/120, Loss: 1.2275
Epoch 19/120, Loss: 0.9923
Epoch 20/120, Loss: 0.9216
Epoch 21/120, Loss: 0.9022
Epoch 22/120, Loss: 0.8647
Epoch 23/120, Loss: 0.8331
Epoch 24/120, Loss: 0.8335
Epoch 25/120, Loss: 0.8253
Epoch 26/120, Loss: 0.8187
Epoch 27/120, Loss: 0.7707
Epoch 28/120, Loss: 0.6996
Epoch 29/120, Loss: 0.7312
Epoch 30/120, Loss: 0.6668
Epoch 31/120, Loss: 0.6072
Epoch 32/120, Loss: 0.6001
Epoch 33/120, Loss: 0.5684
Epoch 34/120, Loss: 0.5362
Epoch 35/120, Loss: 0.5178
Epoch 36/120, Loss: 0.5122
Epoch 37/120, Loss:

In [12]:
print("Evaluasi model pada data testing...")
pred = model(Tensor(X_test, ms.float32)).asnumpy()

# Reshape kembali ke format 3 dimensi (n_samples, forecast_step, n_features)
pred = pred.reshape(-1, forecast, 5)
y_test_reshaped = y_test.reshape(-1, forecast, 5)

# Flatten untuk proses inverse scaling
pred_flat = pred.reshape(-1, 5)
y_test_flat = y_test_reshaped.reshape(-1, 5)

# Dummy array untuk membantu proses inverse transform karena scaler memakai seluruh fitur input
dummy_pred = np.zeros((len(pred_flat), len(fitur_input)))
dummy_pred[:, :5] = pred_flat
y_pred_inv = scaler.inverse_transform(dummy_pred)[:, :5]

dummy_test = np.zeros((len(y_test_flat), len(fitur_input)))
dummy_test[:, :5] = y_test_flat
y_test_inv = scaler.inverse_transform(dummy_test)[:, :5]

# Print metrics per fitur sensor
for i, col in enumerate(fitur_sensor):
    mae = mean_absolute_error(y_test_inv[:, i], y_pred_inv[:, i])
    rmse = np.sqrt(mean_squared_error(y_test_inv[:, i], y_pred_inv[:, i]))
    r2 = r2_score(y_test_inv[:, i], y_pred_inv[:, i])

    print(f"{col:11} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

Evaluasi model pada data testing...
Suhu        | MAE: 1.2958 | RMSE: 1.7714 | R2: 0.7611
Kelembaban  | MAE: 5.1591 | RMSE: 6.5422 | R2: 0.6879
CO2         | MAE: 0.0016 | RMSE: 0.0020 | R2: 0.0000
NH3         | MAE: 0.0325 | RMSE: 0.0406 | R2: -2.7369
Cahaya      | MAE: 0.0012 | RMSE: 0.0016 | R2: -51467900891661732937728.0000


In [13]:
print("\nMenyimpan model & scaler lokal...")
ms.save_checkpoint(model, "./model_lstm.ckpt")
joblib.dump(scaler, "./scaler.save")

print("Mengunggah output ke Huawei Cloud OBS...")
mox.file.copy("./model_lstm.ckpt", "obs://mindspore-dataset-1/output_model2/model_lstm.ckpt")
mox.file.copy("./scaler.save", "obs://mindspore-dataset-1/output_model2/scaler.save")

print("✅ Model & scaler berhasil diperbarui di OBS!")


Menyimpan model & scaler lokal...
Mengunggah output ke Huawei Cloud OBS...
✅ Model & scaler berhasil diperbarui di OBS!
